In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama

import os

C:\Users\gagan\AppData\Local\Temp\ipykernel_30164\339281974.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_path = "dataset/medical_imaging_handbook.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 41


In [3]:
print(documents[0].page_content[:1000])

MEDICAL  MEDICINE  ACCORDING  TO  DISEASE  
HANDBOOK
 
Comprehensive  Clinical  Reference  Guide  
Sixth  Edition  |  2026  
 
TABLE  OF  CONTENTS  
Chapter  1:  Cardiovascular  Diseases  
Chapter
 
2:
 
Respiratory
 
Disorders
 
Chapter
 
3:
 
Endocrine
 
&
 
Metabolic
 
Diseases
 
Chapter
 
4:
 
Neurological
 
Conditions
 
Chapter
 
5:
 
Gastrointestinal
 
Disorders
 
Chapter
 
6:
 
Infectious
 
Diseases
 
Chapter
 
7:
 
Autoimmune
 
&
 
Rheumatologic
 
Diseases
 
Chapter
 
8:
 
Psychiatric
 
&
 
Neurological
 
Disorders
 
Chapter
 
9:
 
Oncology
 
&
 
Hematology
 
Chapter
 
10:
 
Dermatological
 
Conditions
 
Chapter
 
11:
 
Renal
 
&
 
Urological
 
Diseases
 
Chapter
 
12:
 
Musculoskeletal
 
Disorders
 
Appendix
 
A:
 
Drug
 
Interactions
 
&
 
Contraindications
 
Appendix
 
B:
 
Laboratory
 
Reference
 
Ranges
 
Appendix
 
C:
 
Emergency
 
Protocols
 
 
CHAPTER  1:  CARDIOVASCULAR  DISEASES  
1.1  HYPERTENSION


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 75


In [5]:
print(chunks[0].page_content)

MEDICAL  MEDICINE  ACCORDING  TO  DISEASE  
HANDBOOK
 
Comprehensive  Clinical  Reference  Guide  
Sixth  Edition  |  2026  
 
TABLE  OF  CONTENTS  
Chapter  1:  Cardiovascular  Diseases  
Chapter
 
2:
 
Respiratory
 
Disorders
 
Chapter
 
3:
 
Endocrine
 
&
 
Metabolic
 
Diseases
 
Chapter
 
4:
 
Neurological
 
Conditions
 
Chapter
 
5:
 
Gastrointestinal
 
Disorders
 
Chapter
 
6:
 
Infectious
 
Diseases
 
Chapter
 
7:
 
Autoimmune
 
&
 
Rheumatologic
 
Diseases
 
Chapter
 
8:
 
Psychiatric
 
&
 
Neurological
 
Disorders
 
Chapter
 
9:
 
Oncology
 
&
 
Hematology
 
Chapter
 
10:
 
Dermatological
 
Conditions
 
Chapter
 
11:
 
Renal
 
&
 
Urological
 
Diseases
 
Chapter
 
12:
 
Musculoskeletal
 
Disorders
 
Appendix
 
A:
 
Drug
 
Interactions
 
&
 
Contraindications
 
Appendix
 
B:


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8773.98it/s]


In [7]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

In [8]:
vector_db.persist()

C:\Users\gagan\AppData\Local\Temp\ipykernel_30164\2766552738.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


In [9]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

In [12]:
query = "How to fix ACNE?"

results = retriever.invoke(query)

In [13]:
for i, doc in enumerate(results):
    print("="*80)
    print("Result", i+1)
    print("Page:", doc.metadata["page"])
    print()
    print(doc.page_content[:700])

Result 1
Page: 28

AML  Myeloid  blasts  >  20%  Chemotherapy  (7+3),  stem  cell  transplant  
ALL  Lymphoid  blasts;  children  Vincristine,  Prednisone,  Asparaginase  
CML  Philadelphia  chromosome  t(9;22)  
Tyrosine  kinase  inhibitors  (Imatinib)  
CLL  B-cell  proliferation;  older  adults  BTK  inhibitors  (Ibrutinib,  Zanubrutinib),  venetoclax,  rituximab  
 
CHAPTER  10:  DERMATOLOGICAL  CONDITIONS  
10.1  ACNE  
Deﬁnition:  
Disorder
 
of
 
pilosebaceous
 
units
 
causing
 
comedones,
 
papules,
 
pustules,
 
nodules.
 
Treatment  Algorithm:  
●  Mild:  Topical  benzoyl  peroxide,  salicylic  acid,  retinoids  (Adapalene)  ●  Moderate:  Combination  (Benzoyl  peroxide  +  Clindamycin)  OR  (Adapal
Result 2
Page: 28

Benzoyl
 
peroxide)
 ●  Severe:  Oral  antibiotics  (Doxycycline,  Minocycline,  Sarecycline)  +  Topical  
retinoids
 ●  Severe  (Refractory):  Isotretinoin  0.5-1  mg/kg/day  x  4-6  months  (requires  
monitoring
 
for
 
teratogenicity,
 
lipids,
 
mood
 
ch

In [14]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [15]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful medical AI assistant.

Answer ONLY using the provided context.

If the answer is not present in the context,
say:

"I could not find that information in the document."

Context:
{context}

Question:
{question}

Answer:
""")

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

parser = StrOutputParser()

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | parser
)

In [17]:
question = "How to fix ACNE?"

answer = rag_chain.invoke(question)

print(answer)

 Treatment for Acne can vary based on the severity. Here are some options mentioned in the document:

1. Mild acne: Topical benzoyl peroxide, salicylic acid, retinoids (Adapalene)
2. Moderate acne: Combination (Benzoyl peroxide + Clindamycin) OR (Adapalene + Benzoyl peroxide)
3. Severe acne: Oral antibiotics (Doxycycline, Minocycline, Sarecycline) + Topical retinoids
4. Severe (Refractory) acne: Isotretinoin 0.5-1 mg/kg/day for 4-6 months (requires monitoring for teratogenicity, lipids, mood changes)
5. Hormonal Therapy (for females): Spironolactone (anti-androgen)
